# Task 4 — four Tableau dashboards and publication evidence

**Run after Tasks 1–3 produce observed results.** Generate small, suppressed **aggregate** CSV inputs, inspect them and publish one genuine four-dashboard Tableau Public workbook. The notebook does not claim to create the workbook or fabricate its screenshots.

## 1️⃣ SPARK SESSION + MEASURED PREREQUISITE CHECK

Use your verified allocation and cluster; read only the prior **observed** task results before exporting anything to Tableau Public.

In [ ]:
from pathlib import Path
import sys
from pyspark.sql import SparkSession
ROOT = Path.cwd()
if not (ROOT / "coursework").is_dir():
    ROOT = ROOT.parent  # also works when Jupyter starts inside notebooks/
if not (ROOT / "coursework").is_dir():
    raise RuntimeError("Launch Jupyter from the TR-04 project root or notebooks/ directory")
sys.path.insert(0, str(ROOT))
from coursework.settings import (load_config, make_spark, project_path,
    read_json, require_verified_allocation, spark_configuration)
cfg = load_config()  # gitignored config/config.json: your real allocation and cluster
require_verified_allocation(cfg)  # requires YOUR independently checked Aula row and terms
spark = make_spark(cfg, "Task4")  # config-driven SparkSession.builder, not guessed Colab resources
assert isinstance(spark, SparkSession)
print("Student and allocated dataset:", cfg["student"], cfg["allocation"]["pool_reference"],
      cfg["allocation"]["dataset_name"])
print("ACTUAL Spark application / resources:", spark_configuration(spark))
for number in (1, 2, 3):
    prior = read_json(project_path(cfg, "results_dir") / f"task{number}.json")
    assert prior["status"] == "observed", f"Task {number} is not complete"
print("Tasks 1–3 have real observed result JSONs.")


## 2️⃣ SPARK AGGREGATIONS + TABLEAU-SAFE EXPORTS

`run_task4` does groupBy over the *real* staged Spark Parquet and writes quality, model, business and scalability CSVs; borough-hour cells below the configured threshold are suppressed. First run is normally `pending_publication_or_screenshots` until YOU make the actual Tableau Public workbook.

In [ ]:
from coursework.task4 import run_task4, DASHBOARDS
observed_4 = run_task4(spark, cfg)
for i, name in enumerate(DASHBOARDS, 1):
    print(f"Dashboard {i}: {name}")
print("Exports from actual task results and Spark aggregates:", observed_4["exports"])
print("Publication status (pending until verified):", observed_4["status"])


## 3️⃣ CSV FILE MANIFEST + SMALL AGGREGATE PREVIEW

Inspect exported field names and the FIRST aggregated row only; no trip-level records or model predictions should be imported into a PUBLIC Tableau workbook. Review small-cell suppression across filter combinations before publishing.

In [ ]:
import csv
for name in observed_4["exports"]:
    path = project_path(cfg, "results_dir") / name
    with path.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        header, first_aggregate = reader.fieldnames, next(reader, None)
    print(name, "bytes:", path.stat().st_size,
          "columns:", header, "FIRST AGGREGATE:", first_aggregate)


## 4️⃣ DASHBOARD 1 — DATA QUALITY + PIPELINE

In Tableau, show actual rows retained, source GiB, partitions, monthly counts and separately presented non-additive quality flags. This cell previews the real aggregation in a notebook; it is **not** a substitute for the Tableau dashboard.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
quality = pd.read_csv(project_path(cfg, "results_dir") / "tbl_d1_quality.csv")
monthly = quality[(quality["section"] == "month") & (quality["metric"] == "clean_rows")]
monthly = monthly.sort_values("key")
display(monthly)
ax = monthly.plot.bar(x="key", y="value", legend=False, color="#315c84", figsize=(8, 3))
ax.set(title="Clean card-only trips per pickup month", xlabel="2019 month", ylabel="Trips")
plt.tight_layout(); plt.show()


## 5️⃣ DASHBOARD 2 — FOUR-MODEL PERFORMANCE + IMPORTANCE

Show four aligned PR-AUC, ROC-AUC, F1 and CV/full-refit time comparisons, confusion cells, ROC/PR curves, separate **global** tree importance and **local** one-case LIME. All held-out metrics use the Nov–Dec cohort.

In [ ]:
performance = pd.read_csv(project_path(cfg, "results_dir") / "tbl_d2_performance.csv")
importance = pd.read_csv(project_path(cfg, "results_dir") / "tbl_d2_global_tree_importance.csv")
display(performance[["model", "family", "auc_pr", "auc_roc", "positive_f1",
                     "cv_seconds", "refit_seconds"]])
display(importance.head(10))
ax = performance.plot.bar(x="model", y=["auc_pr", "auc_roc"],
                          figsize=(9, 3.5), color=["#168a85", "#315c84"])
ax.set(title="Measured Nov–Dec AUC by model", xlabel="Classifier", ylabel="AUC")
plt.tight_layout(); plt.show()


## 6️⃣ DASHBOARD 3 — BUSINESS INSIGHTS (DESCRIPTIVE ONLY)

Explore borough × pickup hour high-tip recorded rates for one month plus retrospective distance and observed/predicted borough rates; model inputs still exclude trip distance. Never claim that borough differences are causal or that an unknown future cash trip has an observed electronic tip.

In [ ]:
business = pd.read_csv(project_path(cfg, "results_dir") / "tbl_d3_business.csv")
chosen_month = sorted(business["month"].unique())[-1]
shown = business[(business["month"] == chosen_month) &
                 (business["n"] >= int(cfg["model"]["minimum_borough_size"]))]
print("One-month aggregated preview:", chosen_month)
display(shown.sort_values("n", ascending=False).head(8))
heat = shown.pivot(index="borough", columns="pickup_hour", values="high_tip_rate")
fig, ax = plt.subplots(figsize=(10, 3.5))
# NaNs remain blank for suppressed cells; never colour them as zero rates.
img = ax.imshow(heat.to_numpy(dtype=float), aspect="auto", cmap="YlGnBu", vmin=0, vmax=1)
ax.set_yticks(range(len(heat.index)), heat.index)
ax.set_xticks(range(len(heat.columns)), heat.columns)
ax.set(xlabel="Pickup hour", ylabel="Borough",
       title=f"Recorded high-tip rate — {chosen_month} (blank cells suppressed)")
fig.colorbar(img, ax=ax, label="Rate")
plt.tight_layout(); plt.show()


## 7️⃣ DASHBOARD 4 — SCALABILITY + OPTIONAL EVIDENCED COST

Display measured training, shuffle-count, cache and groupBy sample-fraction times. Price estimates appear only if you supply a verifiable hourly cluster price and source; never present a shuffle benchmark as hardware scaling.

In [ ]:
scale = pd.read_csv(project_path(cfg, "results_dir") / "tbl_d4_scalability.csv")
display(scale[["experiment", "model", "config", "wall_seconds", "usd_estimate"]].head(16))
train_times = scale[scale["experiment"] == "training"]
ax = train_times.plot.bar(x="model", y="wall_seconds", legend=False,
                          figsize=(8, 3), color="#dc784d")
ax.set(title="Actual CV + refit duration", xlabel="Spark classifier", ylabel="Seconds")
plt.tight_layout(); plt.show()
print("Cost source:", cfg["tableau"].get("cluster_cost_source") or "No price entered")


## 8️⃣ BUILD ONE FOUR-DASHBOARD TABLEAU PUBLIC WORKBOOK

Use `tableau/BUILD_DASHBOARDS.md` as a construction guide. Publish **your own** workbook with the four named dashboards above, open the public URL while signed out, and check each view. Export four genuine screenshots to `results/tableau_dashboard_1.png` through `_4.png`. Enter the verified public URL and confirmation flag into your gitignored config. A Python HTTP check cannot prove four dashboards exist.

In [ ]:
from coursework.task4 import verify_public_link
url = cfg["tableau"].get("public_workbook_url", "")
print("Actual live Tableau Public URL check:", verify_public_link(url))
print("Real four-dashboard screenshot inventory:",
      [(i, (project_path(cfg, "results_dir") / f"tableau_dashboard_{i}.png").is_file())
       for i in range(1, 5)])
print("Current publication status:", observed_4["status"])


## 9️⃣ REVERIFY PUBLICATION + REAL FOUR-PANEL CONTACT SHEET

After you publish and capture the four genuine dashboards, reload the local config in case you edited it since Step 1, then rerun Task 4 to verify the URL and assemble the **real screenshots**. Do not substitute generated mock images. A pending status is expected if publication is unfinished.

In [ ]:
from IPython.display import Image, display
cfg = load_config()
images_ready = all((project_path(cfg, "results_dir") /
                    f"tableau_dashboard_{i}.png").is_file() for i in range(1, 5))
if cfg["tableau"].get("published_four_dashboards_verified") and images_ready:
    observed_4 = run_task4(spark, cfg)
else:
    print("Publication not complete: retain pending status; no contact sheet invented.")
print("Task 4 status:", observed_4["status"], observed_4["link_check"])
sheet = project_path(cfg, "results_dir") / "task4_dashboard_contact_sheet.png"
if sheet.is_file() and observed_4["status"] == "observed":
    display(Image(filename=str(sheet)))
else:
    print("No genuine verified four-dashboard contact sheet yet.")


## 🔟 THREE INSIGHTS + FINAL SUBMISSION CHECK

Write three **your-own-words** business observations, each with a real chart/number and an appropriate decision and uncertainty. Capture meaningful private-org commits, a reviewed Word report, four runnable notebooks and an honest AI-use declaration. The checker cannot certify a grade or the authenticity of evidence.

In [ ]:
print("Four dashboard themes:", DASHBOARDS)
print("Verified workbook URL:", observed_4["tableau_public_link"])
print("Observed completion status (must be 'observed' before submission):", observed_4["status"])
print("Write each insight as: observation + specific measured figure + decision + caveat.")
print("Then run: python scripts/check_submission.py --report path/to/reviewed.docx")


**Task 4 completion:** the four Tableau Public dashboards, URL, screenshots, contact sheet, report and commits must be genuine student work. An unexecuted notebook or a workbook construction guide does not constitute publication.